<a id="top"></a>

# Lab L0705: tool failures are messages, not exceptions

```yaml
title: "Lab L0705: tool failures are messages, not exceptions"
keywords: tool failure, dispatch, exception to tool_result, is_error, structured error, unknown tool, recovery, offline, lab
estimated duration: 35
```

> **Lesson:** L07. See [objectives.md](../../../../docs/origin/lesson_roadmaps/L07/objectives.md). Solutions: `L0705_lab_solutions.ipynb`. **Pure Python — no API key needed.** You build the loop's failure-handling layer against a *scripted stub model*, so a tool that raises is exercised deterministically.
>
> **Builds on [L05](../L05/objectives.md):** L05 taught the *tool author* what to **return** on failure (errors as data). This lab is the *loop* layer — what to do when the tool **can't even return** (it raises, or the name is unknown).
>
> **After this lab you can:** convert a tool exception into a `tool_result` with `is_error=True`, pass a structured (already-handled) error through unchanged, and watch the model recover — all without crashing the loop.

## Contents

- [1. Setup](#1-setup)
- [2. Problem 1 — Write dispatch: turn a raise into a tool_result](#2-problem-1--write-dispatch-turn-a-raise-into-a-tool_result)
- [3. Problem 2 — The three failure modes, one by one](#3-problem-2--the-three-failure-modes-one-by-one)
- [4. Problem 3 — Watch the model recover (no crash)](#4-problem-3--watch-the-model-recover-no-crash)
- [5. Problem 4 — Why not dump the traceback? (written)](#5-problem-4--why-not-dump-the-traceback-written)
- [6. Problem 5 — Should the loop auto-retry? (written)](#6-problem-5--should-the-loop-auto-retry-written)
- [7. Self-check](#7-self-check)

## 1. Setup

Given: the same tools (`calculator` raises `ValueError`, `lookup` raises `KeyError` on a missing city), the `FakeModel` scaffolding, the `RunResult` dataclass, and — already complete — the `run_loop` you wrote in [L0704](L0704_lab_empty.ipynb). You write only `dispatch`, the loop's failure-handling layer.

In [ ]:
from types import SimpleNamespace


def calculator(expression: str) -> str:
    """Evaluate a simple arithmetic expression, or raise ValueError on non-arithmetic input."""
    allowed = set("0123456789+-*/(). ")
    if not expression or set(expression) - allowed:
        raise ValueError(f"refusing to evaluate non-arithmetic expression: {expression!r}")
    return str(eval(expression))


POPULATIONS = {"tokyo": "37000000", "lagos": "15000000", "paris": "11000000"}


def lookup(city: str) -> str:
    """Return a city's population, or raise KeyError if it isn't in the table."""
    key = city.strip().lower()
    if key not in POPULATIONS:
        raise KeyError(f"no population on file for {city!r}")
    return POPULATIONS[key]


# Dispatch table: tool NAME -> the function that runs it.
TOOLS: dict[str, object] = {"calculator": calculator, "lookup": lookup}

In [ ]:
def text_block(s: str) -> SimpleNamespace:
    """A fake assistant text block."""
    return SimpleNamespace(type="text", text=s)


def tool_use_block(call_id: str, name: str, args: dict[str, object]) -> SimpleNamespace:
    """A fake assistant tool_use block (mimics the SDK's tool_use content block)."""
    return SimpleNamespace(type="tool_use", id=call_id, name=name, input=args)


def response(blocks: list[SimpleNamespace]) -> SimpleNamespace:
    """A fake model response."""
    stop = "tool_use" if any(b.type == "tool_use" for b in blocks) else "end_turn"
    return SimpleNamespace(content=blocks, stop_reason=stop)


class FakeModel:
    """A scripted stand-in for the Anthropic client: .create() pops the next response."""

    def __init__(self, scripted: list[SimpleNamespace]) -> None:
        self._scripted = scripted
        self.calls = 0

    def create(self, **kwargs: object) -> SimpleNamespace:
        # If the script runs out, reuse the last line -- this is how we simulate a runaway.
        i = min(self.calls, len(self._scripted) - 1)
        self.calls += 1
        return self._scripted[i]

In [ ]:
from dataclasses import dataclass


@dataclass
class RunResult:
    """The answer, the number of model calls, and WHY the loop stopped."""

    final_text: str
    iterations: int
    termination: str  # "natural" or "max_steps"

In [ ]:
def run_loop(model: object, tools: dict[str, object], user_msg: str, max_steps: int) -> RunResult:
    """The loop from L0704 (given here). It calls dispatch() -- which YOU write below."""
    messages: list[dict[str, object]] = [{"role": "user", "content": user_msg}]
    for step in range(1, max_steps + 1):
        resp = model.create(
            model="claude-sonnet-4-6", max_tokens=512, tools=list(tools), messages=messages
        )
        tool_uses = [b for b in resp.content if b.type == "tool_use"]
        if not tool_uses:
            text = "".join(b.text for b in resp.content if b.type == "text")
            return RunResult(text, step, "natural")
        messages.append({"role": "assistant", "content": resp.content})
        results = [dispatch(tools, call) for call in tool_uses]
        messages.append({"role": "user", "content": results})
    return RunResult("", max_steps, "max_steps")

[↑ Back to top](#top)

## 2. Problem 1 — Write dispatch: turn a raise into a tool_result

`dispatch(tools, call)` runs one requested tool and **always returns a `tool_result` dict** — it never lets an exception escape. Three cases:

1. **Unknown tool name** (`tools.get(call.name)` is `None`): return a `tool_result` with `is_error=True` and content like `no such tool 'X'`.
2. **Tool runs cleanly:** return `{"type": "tool_result", "tool_use_id": call.id, "content": output}`.
3. **Tool raises:** catch the exception and return a `tool_result` with `is_error=True` and `content=repr(exc)` — a **short** message (class + one line), **never** a traceback.

Use `fn(**call.input)` to call the tool with the model's arguments.

In [ ]:
def dispatch(tools: dict[str, object], call: object) -> dict[str, object]:
    # TODO case 1: unknown name -> tool_result(is_error=True, 'no such tool ...').
    # TODO case 2: success    -> tool_result with the tool's output.
    # TODO case 3: tool raised -> tool_result(is_error=True, content=repr(exc)).
    raise NotImplementedError("your code here")


ok = tool_use_block("a", "lookup", {"city": "Paris"})
print(dispatch(TOOLS, ok))  # expect a tool_result with content '11000000', no is_error

[↑ Back to top](#top)

## 3. Problem 2 — The three failure modes, one by one

Run `dispatch` on three crafted calls and confirm each becomes a well-formed `tool_result` rather than a crash:

1. a `lookup` for a city **not in the table** (the tool **raises** `KeyError`),
2. a `calculator` with **non-arithmetic** input (the tool **raises** `ValueError`),
3. a call to a tool that **doesn't exist** (`wikipedia`).

Print each result; each should carry `is_error: True`.

In [ ]:
bad_calls = [
    tool_use_block("b1", "lookup", {"city": "Atlantis"}),  # KeyError
    tool_use_block("b2", "calculator", {"expression": "two + 2"}),  # ValueError
    tool_use_block("b3", "wikipedia", {"query": "geese"}),  # unknown tool
]
# TODO: print dispatch(TOOLS, call) for each; confirm each has is_error == True.
raise NotImplementedError("your code here")

[↑ Back to top](#top)

## 4. Problem 3 — Watch the model recover (no crash)

Script a stub that first looks up a missing city (your `dispatch` turns the `KeyError` into an `is_error` `tool_result`), then — having 'seen' the error — looks up a city that exists, then answers. Run `run_loop` and confirm it reaches **natural** termination instead of crashing. The loop turned a tool bug into a message the model could act on.

In [ ]:
recover_script = [
    response([tool_use_block("f1", "lookup", {"city": "Atlantis"})]),  # raises -> is_error
    response([tool_use_block("f2", "lookup", {"city": "Tokyo"})]),  # recover
    response([text_block("Atlantis isn't on file; Tokyo is about 37,000,000.")]),
]
# TODO: model = FakeModel(recover_script); run_loop(..., max_steps=10);
#       assert termination == 'natural'.
raise NotImplementedError("your code here")

[↑ Back to top](#top)

## 5. Problem 4 — Why not dump the traceback? (written)

Your `dispatch` feeds the model `repr(exc)` — a short message — not the full Python traceback. Give **two** reasons that is the right call. (Hint: think about tokens, what the model can act on, and what a stack trace might leak.)

_Write your answer by editing this cell (double-click)._

1. _TODO_
2. _TODO_

[↑ Back to top](#top)

## 6. Problem 5 — Should the loop auto-retry? (written)

Your loop surfaces the error to the model and lets *it* decide whether to retry — the loop does **not** auto-retry. In a sentence or two: why is blanket auto-retry a poor default? (Hint: a `404 not found` is not a `503 service unavailable`; consider a tool that charges a card.)

_Write your answer by editing this cell (double-click)._

_TODO_

[↑ Back to top](#top)

## 7. Self-check

Compare against `L0705_lab_solutions.ipynb`. Done when: `dispatch` returns a clean `tool_result` for the good call and an `is_error` `tool_result` (never a crash) for the missing city, the bad expression, and the unknown tool; and the recovery run reaches `"natural"` termination. You can state why a short error beats a traceback and why auto-retry is not a safe default.

[↑ Back to top](#top)